In [3]:
import pandas as pd
import numpy as np
import json
import os
from jiwer import wer, cer
from datasets import load_dataset
import re
from num2words import num2words
from scipy.stats import iqr, ttest_rel
import matplotlib.pyplot as plt
import seaborn as sns



Analysis of the Wrist Angel Dataset being run through the ASR module only.

In [12]:
batches_per_epoch = 1689
batch_stats = []
with open("../results/wav2vec2_baseline.jsonl") as f:
    for batch_idx, line in enumerate(f):
        samples = json.loads(line)

        avg_duration = sum(s["seg_end"] for s in samples) / len(samples)

        batch_stats.append({
            "epoch": batch_idx // batches_per_epoch,
            "batch": batch_idx % batches_per_epoch,
            "avg_audio_duration": avg_duration,
            "n_samples": len(samples),
        })

df = pd.DataFrame(batch_stats)
df.head(20)

,epoch,batch,avg_audio_duration,n_samples
0,0,0,2.6320,5
1,0,1,7.1760,5
2,0,2,7.1480,5
3,0,3,12.0360,5
4,0,4,12.6080,5
5,0,5,6.4920,5
6,0,6,3.9000,5
7,0,7,5.7200,5
8,0,8,6.4000,5
9,0,9,4.1120,5


In [ ]:
import jiwer

def evaluate_wer_components(references: list[str], hypotheses: list[str]):
    """
    Computes global WER alongside its individual error constituents:
    Substitutions (S), Insertions (I), Deletions (D), and Reference Word Count (N).
    """
    # Process transcriptions across the dataset
    word_output = jiwer.process_words(references, hypotheses)
    
    # Extract raw counts across all audio files
    w_sub = word_output.substitutions
    w_ins = word_output.insertions
    w_del = word_output.deletions
    w_total = word_output.hits + word_output.substitutions + word_output.deletions # N
    wer = word_output.wer

    char_output = jiwer.process_characters(references, hypotheses)
    
    c_sub = char_output.substitutions
    c_ins = char_output.insertions
    c_del = char_output.deletions
    c_total = char_output.hits + c_sub + c_del  # Total reference characters (N_chars)
    cer = char_output.cer
    
    print("=" * 45)
    print("           ASR ERROR EVALUATION           ")
    print("=" * 45)
    print(f"WORD LEVEL (WER: {wer:.2%})")
    print(f"  • Reference Words (N) : {wer}")
    print(f"  • Substitutions (S)   : {w_sub} ({w_sub / w_total:.2%})")
    print(f"  • Insertions (I)      : {w_ins} ({w_ins / w_total:.2%})")
    print(f"  • Deletions (D)       : {w_del} ({w_del / w_total:.2%})")
    print("-" * 45)
    print(f"CHARACTER LEVEL (CER: {cer:.2%})")
    print(f"  • Reference Chars (N) : {c_total}")
    print(f"  • Substitutions (S)   : {c_sub} ({c_sub / c_total:.2%})")
    print(f"  • Insertions (I)      : {c_ins} ({c_ins / c_total:.2%})")
    print(f"  • Deletions (D)       : {c_del} ({c_del / c_total:.2%})")
    print("=" * 45)

    return {
        "wer": wer,
        "cer": cer,
        "word_metrics": {
            "substitutions": w_sub,
            "insertions": w_ins,
            "deletions": w_del,
            "total_words": w_total,
        },
        "char_metrics": {
            "substitutions": c_sub,
            "insertions": c_ins,
            "deletions": c_del,
            "total_chars": c_total,
        }
    }

In [22]:
references = []
hypotheses = []

with open("../results/wav2vec2_baseline.jsonl") as f:
    for line in f:
        batch = json.loads(line)
        for item in batch:
            references.append(item['ref'])
            hypotheses.append(item['hyp'])
print(f'Ref shape: {len(references)}')
print(f'Hyp length: {len(hypotheses)}')

evaluate_wer_components(references=references, hypotheses=hypotheses)

Ref shape: 25314
Hyp length: 25314
           ASR ERROR EVALUATION           
WORD LEVEL (WER: 34.30%)
  • Reference Words (N) : 0.34299239856938785
  • Substitutions (S)   : 68598 (23.84%)
  • Insertions (I)      : 11061 (3.84%)
  • Deletions (D)       : 19023 (6.61%)
---------------------------------------------
CHARACTER LEVEL (CER: 16.43%)
  • Reference Chars (N) : 1341279
  • Substitutions (S)   : 83613 (6.23%)
  • Insertions (I)      : 42108 (3.14%)
  • Deletions (D)       : 94656 (7.06%)


{'wer': 0.34299239856938785,
 'cer': 0.1643036236308777,
 'word_metrics': {'substitutions': 68598,
  'insertions': 11061,
  'deletions': 19023,
  'total_words': 287709},
 'char_metrics': {'substitutions': 83613,
  'insertions': 42108,
  'deletions': 94656,
  'total_chars': 1341279}}

## Computing Sentence Semantic Distance
As part of the transcript accuracy evaluation, I will compute a more context-based accuracy evaluation in order to provide a more nuanced perspective on the correctness of the generated transcripts.
It will be using a standard multilingual sentence transformer for all SemDist evaluations to ensure consistency in the results.

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util

def compute_sentence_semantic_distance(
    references: list[str], 
    hypotheses: list[str], 
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
) -> list[float]:
    """
    Calculates the Sentence Semantic Distance (1 - Cosine Similarity) 
    between reference and ASR hypothesis transcripts.
    """
    # Load Sentence Transformer model
    model = SentenceTransformer(model_name)
    
    # 1. Compute dense embeddings for references and hypotheses
    ref_embeddings = model.encode(references, convert_to_tensor=True, show_progress_bar=False)
    hyp_embeddings = model.encode(hypotheses, convert_to_tensor=True, show_progress_bar=False)
    
    # 2. Calculate pairwise cosine similarity along the diagonal
    # util.cos_sim returns a similarity matrix of shape [N, N]
    cosine_sims = util.cos_sim(ref_embeddings, hyp_embeddings).diagonal()
    
    # 3. Convert similarity to distance: Distance = 1 - Similarity
    semantic_distances = 1.0 - cosine_sims.cpu().numpy()
    
    # Clip small negative floating-point errors around zero
    semantic_distances = np.clip(semantic_distances, 0.0, 2.0)
    
    return semantic_distances.tolist()


/root/master_thesis/thesis_multi_speaker_asr/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


REF: recepten blev fornyet af lægen
HYP: lægen fornyede recepten
Semantic Distance: 0.0262

REF: patienten udskrives i dag
HYP: patienten indlægges i dag
Semantic Distance: 0.0799



In [ ]:
wav2vec2_hyp = []
wav2vec2_ref = []
with open("../results/wav2vec2_baseline.jsonl", "r", encoding="utf-8") as file:
    iter = 1
    for line in file:
        batch = json.loads(line)
        for sample in batch:
            # Look for epoch metadata (adjust key to match your logger)
            if iter >= 1689:
                wav2vec2_hyp.append(sample['hyp'])
                wav2vec2_ref.append(sample['ref'])
        iter += 1

distances = compute_sentence_semantic_distance(references=wav2vec2_ref, hypotheses=wav2vec2_hyp)
for r, h, d in zip(wav2vec2_ref, wav2vec2_hyp, distances):
    print(f"REF: {r}")
    print(f"HYP: {h}")
    print(f"Semantic Distance: {d:.4f}\n")

# For evaluating speaker-based accuracy for Wrist Angel Dataset

In [ ]:
from collections import defaultdict
import jiwer

def evaluate_speaker_level_asr(dataset: list[dict]) -> dict[str, dict]:
    """
    Computes corpus-level (micro-averaged) WER, CER, and detailed error 
    breakdowns for each individual speaker in the dataset.
    
    `dataset` should be a list of dicts formatted as:
      [{"speaker_id": "spk_01", "reference": "...", "hypothesis": "..."}, ...]
    """
    # 1. Group transcripts by speaker_id
    speaker_data = defaultdict(lambda: {"refs": [], "hyps": []})
    for entry in dataset:
        spk = entry["speaker_id"]
        speaker_data[spk]["refs"].append(entry["reference"])
        speaker_data[spk]["hyps"].append(entry["hypothesis"])

    speaker_results = {}

    # 2. Compute micro-averaged metrics for each speaker
    for spk_id, data in speaker_data.items():
        refs = data["refs"]
        hyps = data["hyps"]

        # Word-level (WER)
        w_out = jiwer.process_words(refs, hyps)
        w_sub, w_ins, w_del = w_out.substitutions, w_out.insertions, w_out.deletions
        w_total = w_out.hits + w_sub + w_del

        # Character-level (CER)
        c_out = jiwer.process_characters(refs, hyps)
        c_sub, c_ins, c_del = c_out.substitutions, c_out.insertions, c_out.deletions
        c_total = c_out.hits + c_sub + c_del

        speaker_results[spk_id] = {
            "num_utterances": len(refs),
            "wer": w_out.wer,
            "cer": c_out.cer,
            "word_breakdown": {
                "total_words": w_total,
                "substitutions": w_sub,
                "insertions": w_ins,
                "deletions": w_del,
            },
            "char_breakdown": {
                "total_chars": c_total,
                "substitutions": c_sub,
                "insertions": c_ins,
                "deletions": c_del,
            }
        }

    return speaker_results


def print_speaker_summary(speaker_results: dict[str, dict]):
    """Utility to print a clean summary table of speaker results."""
    print(f"{'Speaker ID':<15} | {'Utterances':<10} | {'WER':<8} | {'CER':<8} | {'Word Errors (S/I/D)':<20}")
    print("-" * 72)
    
    for spk_id, res in speaker_results.items():
        w_b = res["word_breakdown"]
        errors_str = f"{w_b['substitutions']}/{w_b['insertions']}/{w_b['deletions']}"
        print(f"{spk_id:<15} | {res['num_utterances']:<10} | {res['wer']:>6.2%} | {res['cer']:>6.2%} | {errors_str:<20}")




In [26]:
from datasets import Dataset

ds = load_dataset("CoRal-project/coral-v3", "conversation", split='test')
df = Dataset.to_pandas(ds)
id_to_speaker = df.set_index('id_conversation')['id_speaker'].to_dict()
all_samples = []
with open("../results/wav2vec2_baseline.jsonl") as file:
    for line in file:
        batch = json.loads(line)
        for sample in batch:
            audio_id = sample.get('audio_id') 
            # Map speaker_id using dictionary lookup (defaults to None if not found)
            sample['id_speaker'] = id_to_speaker.get(audio_id, None)
            
            all_samples.append(sample)


In [39]:
from collections import defaultdict
import jiwer

def evaluate_speaker_level_asr(dataset: list[dict]) -> dict[str, dict]:
    """
    Computes corpus-level (micro-averaged) WER, CER, and detailed error 
    breakdowns for each individual speaker in the dataset.
    
    `dataset` should be a list of dicts formatted as:
      [{"speaker_id": "spk_01", "reference": "...", "hypothesis": "..."}, ...]
    """
    # 1. Group transcripts by speaker_id
    speaker_data = defaultdict(lambda: {"refs": [], "hyps": []})
    for entry in dataset:
        spk = entry["id_speaker"]
        speaker_data[spk]["refs"].append(entry["ref"])
        speaker_data[spk]["hyps"].append(entry["hyp"])

    speaker_results = {}

    # 2. Compute micro-averaged metrics for each speaker
    for spk_id, data in speaker_data.items():
        refs = data["refs"]
        hyps = data["hyps"]

        # Word-level (WER)
        w_out = jiwer.process_words(refs, hyps)
        w_sub, w_ins, w_del = w_out.substitutions, w_out.insertions, w_out.deletions
        w_total = w_out.hits + w_sub + w_del

        # Character-level (CER)
        c_out = jiwer.process_characters(refs, hyps)
        c_sub, c_ins, c_del = c_out.substitutions, c_out.insertions, c_out.deletions
        c_total = c_out.hits + c_sub + c_del

        speaker_results[spk_id] = {
            "num_utterances": len(refs),
            "wer": w_out.wer,
            "cer": c_out.cer,
            "word_breakdown": {
                "total_words": w_total,
                "substitutions": w_sub,
                "insertions": w_ins,
                "deletions": w_del,
            },
            "char_breakdown": {
                "total_chars": c_total,
                "substitutions": c_sub,
                "insertions": c_ins,
                "deletions": c_del,
            }
        }

    return speaker_results


def print_speaker_summary(speaker_results: dict[str, dict]):
    """Utility to print a clean summary table of speaker results."""
    print(f"{'Speaker ID':<15} | {'Utterances':<10} | {'WER':<8} | {'CER':<8} | {'Word Errors (S/I/D)':<20}")
    print("-" * 72)
    
    for spk_id, res in speaker_results.items():
        w_b = res["word_breakdown"]
        errors_str = f"{w_b['substitutions']}/{w_b['insertions']}/{w_b['deletions']}"
        print(f"{spk_id:<15} | {res['num_utterances']:<10} | {res['wer']:>6.2%} | {res['cer']:>6.2%} | {errors_str:<20}")




In [41]:
import json

steady_state_samples = []

with open("../results/wav2vec2_baseline.jsonl", "r", encoding="utf-8") as file:
    iter = 1
    for line in file:
        batch = json.loads(line)
        for sample in batch:
            # Look for epoch metadata (adjust key to match your logger)
            if iter >= 1689:
                audio_id = sample.get("audio_id")
                sample["id_speaker"] = id_to_speaker.get(audio_id, None)
                steady_state_samples.append(sample)

        iter += 1

In [46]:
speaker_dict = evaluate_speaker_level_asr(steady_state_samples)
print_speaker_summary(speaker_dict)

Speaker ID      | Utterances | WER      | CER      | Word Errors (S/I/D) 
------------------------------------------------------------------------
spe_fa639f5932359117682753884585d883 | 390        | 24.94% | 12.24% | 1088/246/502        
spe_5e319f90767d47e11731d95e314e4670 | 732        | 44.33% | 20.09% | 1786/292/276        
spe_df3293886215084f5fd6a447bb379b11 | 402        | 31.51% | 13.51% | 768/152/152         
spe_4aa23a60464a18e3597cdeb3606ac572 | 1188       | 34.32% | 16.60% | 2648/458/710        
spe_03e8b9d0ee8d3192e113ff62c61e4916 | 812        | 36.24% | 16.82% | 2578/350/756        
spe_deedf738efa054ae460989be3033a3cf | 238        | 23.42% | 10.10% | 410/102/102         
spe_741ba3dd1acd26458718a591a980d743 | 638        | 29.14% | 12.99% | 1802/268/420        
spe_2937b289da4c0a7b9877c56ecead4794 | 566        | 29.38% | 13.27% | 1604/266/342        
spe_fbf3381f525dbe5ddf1a2a1d36e9c4b9 | 756        | 42.45% | 20.10% | 2048/304/388        
spe_6e67cbe51a49d9e4abbd7699a4a89d

In [ ]:
ds = load_dataset("CoRal-project/coral-v3", "conversation", split='test')
df = Dataset.to_pandas(ds)
id_to_speaker = df.set_index('id_conversation')['id_speaker'].to_dict()
all_samples = []
with open("../results/wav2vec2_baseline.jsonl") as file:
    for line in file:
        batch = json.loads(line)
        for sample in batch:
            audio_id = sample.get('audio_id') 
            # Map speaker_id using dictionary lookup (defaults to None if not found)
            speaker_id = id_to_speaker.get(audio_id, None)
            speaker_err = speaker_dict.get(speaker_id)
            sample['wer'] = speaker_err['wer']
            sample['cer'] = speaker_err['cer']
            all_samples.append(sample)

## Calculating and evaluating the computational resource related metrics
For this section I will run through computing the different metrics, including; CPU Time, Wall Time, RAM usage, Real-Time-Factor (RTF), Latency and Throughput.

The RAM usage will more so be reported as a descriptive statistic than an actual evaluation. Furthermore, the CPU efficiency will also be estimated in order to assess whether there was/is room for potential improvements if the core utilization was not maximized.

In [39]:
def get_walltime(filename: str):
    epoch_pattern = r'Epoch: (\d+)'
    walltime_pattern = r'Walltime:\s*([+-]?(?:[0-9]*\.)?[0-9]+)'
    cputime_pattern = r'CPU time:\s*([+-]?(?:[0-9]*\.)?[0-9]+)'

    walltime_res = []
    cputime_res = []
    with open(filename, 'r') as file:
        for line in file:
            m_epoch = re.search(epoch_pattern, line)
            m_walltime = re.search(walltime_pattern, line)
            m_cputime = re.search(cputime_pattern, line)

            if not m_epoch or not m_walltime or not m_cputime:
                continue


            walltime = float(m_walltime.group(1))
            cpu_time = float(m_cputime.group(1))
            walltime_res.append(walltime)
            cputime_res.append(cpu_time)

    return walltime_res, cputime_res


In [40]:
def get_durations(filename: str):
    durations = []
    with open(filename, 'r') as file:
        for line in file:
            batch = json.loads(line)
            batch_duration = 0
            for sample in batch:
                sample_duration = sample['seg_end']
                batch_duration += sample_duration
            durations.append(batch_duration)

    return durations

In [ ]:
import pandas as pd

def compute_performance_metrics(
    df: pd.DataFrame, 
    num_cpu_threads: int = 8
) -> pd.DataFrame:
    """
    Computes Wall Time, CPU Time, RTF, CPU Parallelization Efficiency, 
    and RAM Savings relative to a specified baseline model
    """
    # Group metrics by model (averaging across logged runs/batches)
    summary = df.groupby('model_name').agg({
        'wall_time_sec': 'sum',
        'cpu_time_sec': 'sum',
        'total_audio_duration_sec': 'sum'
    }).reset_index()

    # 1. Compute Real-Time Factor (RTF)
    summary['RTF'] = summary['wall_time_sec'] / summary['total_audio_duration_sec']

    # 2. Compute CPU Parallelization Efficiency (%)
    # Ratio of CPU time spent relative to theoretical maximum across allocated threads
    summary['CPU_Efficiency_pct'] = (
        summary['cpu_time_sec'] / (summary['wall_time_sec'] * num_cpu_threads)
    ) * 100


    return summary

### Overall Compute Performance Metrics
#### Roest-v3-wav2vec2-315m - CoRal

In [ ]:
walltime_arr, cputime_arr = get_walltime('../results/wav2vec2_baseline.log')
durations = get_durations('../results/wav2vec2_baseline.jsonl')
model_name = ['roest-v3-wav2vec2-315m'] * len(walltime_arr)
df_compute = pd.DataFrame({'model_name': model_name,'wall_time_sec': walltime_arr, 'cpu_time_sec': cputime_arr, 'total_audio_duration_sec': durations})

metrics_summary = compute_performance_metrics(
    df_compute, 
    num_cpu_threads=6
)

# Display formatted table
print(metrics_summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

#### Roest-v3-wav2vec2-315m - Wrist Angel

In [ ]:
walltime_arr, cputime_arr = get_walltime('../results/wav2vec2_baseline.log')
durations = get_durations('../results/wav2vec2_baseline.jsonl')
model_name = ['roest-v3-wav2vec2-315m'] * len(walltime_arr)
df_compute = pd.DataFrame({'model_name': model_name,'wall_time_sec': walltime_arr, 'cpu_time_sec': cputime_arr, 'total_audio_duration_sec': durations})

metrics_summary = compute_performance_metrics(
    df_compute, 
    num_cpu_threads=6
)

# Display formatted table
print(metrics_summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))